In [1]:
import os
os.chdir("../")
%pwd

'/home/minh_khai/pneumonia-mlops'

In [2]:
from dataclasses import dataclass
from pathlib import Path
import numpy as np

@dataclass
class Experience:
    state:      np.ndarray
    action:     int
    reward:     float
    next_state: np.ndarray
    done:       bool

@dataclass(frozen=True)
class DuelingDQN_Params_Config:
    n_metrics:      int
    output_steps:   int
    action_size:    int
    hidden_size:    int
    
    @property
    def state_size(self) -> int:
        return self.n_metrics * (1 + self.output_steps)
    
@dataclass(frozen=True)
class DQN_Params_Config:
    lr:         float
    gamma:      float
    epsilon:    float
    eps_min:    float
    eps_decay:  float
    batch_size: int
    target_update_freq: int

@dataclass(frozen=True)
class DQN_Planner_Config:
    root_dir:   Path
    model_dir:  Path
    
    duel_dqn_params:    DuelingDQN_Params_Config
    dqn_params:         DQN_Params_Config

@dataclass(frozen=True)
class Simulation_Config:
    cpu_warning:    int
    cpu_critical:   int
    ram_warning:    int
    ram_critical:   int
    latency_warning:  float
    latency_critical: float
    drift_warning:  float
    drift_critical: float
    output_steps:   int

In [3]:
from core.constants import *
from core.utils import read_yaml, create_directories

class ConfigManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH, 
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_simulation_config(self) -> Simulation_Config:
        params = self.params.simulation_params
        return Simulation_Config(
            cpu_warning     = params.cpu_warning,
            cpu_critical    = params.cpu_critical,
            ram_warning     = params.ram_warning,
            ram_critical    = params.ram_critical,
            latency_warning = params.latency_warning,
            latency_critical    = params.latency_critical,
            drift_warning   = float(params.drift_warning),
            drift_critical  = float(params.drift_critical),
            output_steps    = params.output_steps
        )
        
    def get_dqn_planner_config(self) -> DQN_Planner_Config:
        config = self.config.dqn_planner_config
        params = self.params.dqn_planner_params
        create_directories([config.root_dir])
        
        return DQN_Planner_Config(
            root_dir  = Path(config.root_dir),
            model_dir = Path(config.root_dir) / "dqn_model.pth",
            
            duel_dqn_params = DuelingDQN_Params_Config(
                n_metrics   = params.n_metrics,
                output_steps= params.output_steps,
                action_size = params.action_size,
                hidden_size = params.hidden_size
            ),
            
            dqn_params = DQN_Params_Config(
                lr         = params.lr,
                gamma      = params.gamma,
                epsilon    = params.epsilon,
                eps_min    = params.epsilon_min,
                eps_decay  = params.epsilon_decay,
                batch_size = params.batch_size,
                target_update_freq = params.target_update_freq
            )
        )

In [4]:
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity: int = 10000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, experience: Experience):
        self.buffer.append(experience)
    
    def sample(self, batch_size: int) -> list[Experience]:
        return random.sample(self.buffer, batch_size)
    
    def __len__(self):
        return len(self.buffer)

In [5]:
import torch.nn as nn

# Q(s,a) = V(s) + A(s,a) - mean(A(s,a))
class DuelingDQN(nn.Module):
    def __init__(self, state_size: int, action_size: int, hidden_size: int = 128):
        super().__init__()

        # Shared layers
        self.shared = nn.Sequential(
            nn.Linear(state_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU()
        )
        
        # Value stream V(s)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
        
        # Advantage stream A(s, a)
        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
        
    def forward(self, x):
        shared = self.shared(x)
        value  = self.value_stream(shared)
        advantage = self.advantage_stream(shared)
        return value + advantage - advantage.mean(dim=1, keepdim=True)

In [6]:
import numpy as np

"""
    Simulate state of microservices for DQN training
    State: [
        cpu, ram, latency, drift, 
        predicted_cpu*4, predicted_ram*4, predicted_latency*4, predicted_drift*4
    ]
"""

class SystemSimulation:
    def __init__(self, config: Simulation_Config):
        self.config     = config
        self.state_size = 24
        self.reset()
        
    def reset(self) -> np.ndarray:
        scenario = np.random.choice(
            ['healthy', 'high_drift', 'high_cpu', 'high_load'],
            p=[0.2, 0.5, 0.2, 0.1]
        )
        
        if scenario == 'healthy':
            self.cpu     = np.random.uniform(0.2, 0.5)
            self.ram     = np.random.uniform(0.3, 0.6)
            self.latency = np.random.uniform(0.01, 0.06)
            self.drift   = np.random.uniform(0.0, 0.2)
        elif scenario == 'high_drift':
            self.cpu     = np.random.uniform(0.1, 0.4)
            self.ram     = np.random.uniform(0.3, 0.6)
            self.latency = np.random.uniform(0.01, 0.05)
            self.drift   = np.random.uniform(0.35, 0.8)
        elif scenario == 'high_cpu':
            self.cpu     = np.random.uniform(0.75, 0.95)
            self.ram     = np.random.uniform(0.5, 0.8)
            self.latency = np.random.uniform(0.05, 0.15)
            self.drift   = np.random.uniform(0.0, 0.2)
        elif scenario == 'high_load':
            self.cpu     = np.random.uniform(0.7, 0.9)
            self.ram     = np.random.uniform(0.6, 0.85)
            self.latency = np.random.uniform(0.08, 0.2)
            self.drift   = np.random.uniform(0.2, 0.5)
        
        self.step_n = 0
        return self._get_state()

    def _get_state(self) -> np.ndarray:
        cpu, ram, lat, drift = self.cpu, self.ram, self.latency, self.drift
        current  = [cpu, ram, lat, drift]
        predicted = []
        for _ in range(self.config.output_steps):
            cpu   = np.clip(cpu   + np.random.normal(0, 0.02),  0, 1)
            ram   = np.clip(ram   + np.random.normal(0, 0.01),  0, 1)
            lat   = np.clip(lat   + np.random.normal(0, 0.001), 0, 1)
            drift = np.clip(drift + np.random.normal(0, 0.01),  0, 1)
            predicted += [cpu, ram, lat, drift]
        return np.array(current + predicted, dtype=np.float32)
    
    def step(self, action: int):
        self.step_n  += 1
        self.cpu     += np.random.normal(0, 0.02)
        self.ram     += np.random.normal(0, 0.01)
        self.latency += np.random.normal(0, 0.001)
        self.drift   += np.random.normal(0, 0.01)

        if action == 1:
            self.drift   = max(0, self.drift - 0.3)
        elif action == 2:
            self.ram     = max(0, self.ram - 0.2)
            self.cpu     = max(0, self.cpu - 0.15)
            self.latency += 0.05
        elif action == 3:
            self.latency = max(0, self.latency - 0.1)
            self.ram     += 0.1
        elif action == 4:
            self.latency = max(0, self.latency - 0.1)
            self.ram     += 0.05
        elif action == 5:
            self.latency = max(0, self.latency - 0.15)
            self.cpu     = max(0, self.cpu - 0.1)
        elif action == 6:
            self.latency += 0.05
            self.cpu     += 0.05
        elif action == 7:
            self.latency += 0.1
            self.ram     += 0.05

        self.cpu     = np.clip(self.cpu,     0, 1)
        self.ram     = np.clip(self.ram,     0, 1)
        self.latency = np.clip(self.latency, 0, 1)
        self.drift   = np.clip(self.drift,   0, 1)

        reward = self._compute_reward(action)
        done   = self.step_n >= 200 or \
                self.ram  > self.config.ram_critical or \
                self.cpu  > self.config.cpu_critical
        return self._get_state(), reward, done
    
    def _compute_reward(self, action: int) -> float:
        reward = 0.0

        if self.drift > self.config.drift_critical:
            reward -= 2.0
            if action == 0: reward -= 1.0
        elif self.drift > self.config.drift_warning:
            reward -= 0.8
            if action == 0: reward -= 0.5

        if self.cpu > self.config.cpu_critical:
            reward -= 1.5
            if action == 0: reward -= 0.5
        elif self.cpu > self.config.cpu_warning:
            reward -= 0.5

        if self.ram > self.config.ram_critical:
            reward -= 1.5
            if action == 0: reward -= 0.5
        elif self.ram > self.config.ram_warning:
            reward -= 0.5

        if self.latency > self.config.latency_critical:
            reward -= 1.0
            if action == 0: reward -= 0.3
        elif self.latency > self.config.latency_warning:
            reward -= 0.3

        if action != 0:
            if (self.drift   < self.config.drift_warning and
                self.cpu     < self.config.cpu_warning   and
                self.ram     < self.config.ram_warning   and
                self.latency < self.config.latency_warning):
                reward -= 1.0

        if (self.drift   < self.config.drift_warning and
            self.cpu     < self.config.cpu_warning   and
            self.ram     < self.config.ram_warning   and
            self.latency < self.config.latency_warning):
            reward += 0.05

        return reward

In [15]:
ACTIONS = {
    0: "do_nothing",
    1: "trigger_retraining",
    2: "switch_to_lighter_model",
    3: "scale_up_service",      # legacy
    4: "restart_service",
    5: "scale_out_service",     # new
    6: "scale_in_service",      # new
    7: "swap_model_version",    # new
}

import random, math
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F

from core.logging import logger

class DQNPlanner:
    def __init__(self, config: DQN_Planner_Config):
        self.config = config
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.epsilon = self.config.dqn_params.epsilon
        
        self.online_net = self._get_network().to(self.device)
        self.target_net = self._get_network().to(self.device)
        self.target_net.load_state_dict(self.online_net.state_dict())
        self.target_net.eval()
        
        self.optimizer  = torch.optim.Adam(self.online_net.parameters(), lr=self.config.dqn_params.lr)
        self.buffer     = ReplayBuffer()
        self._ready     = False
        self.step_count = 0
    
    def _get_network(self):
        return DuelingDQN(
            state_size  = self.config.duel_dqn_params.state_size,
            action_size = self.config.duel_dqn_params.action_size,
            hidden_size = self.config.duel_dqn_params.hidden_size
        )
        
    # --------------------------- Inference ------------------------------------
    def load(self):
        if not Path(self.config.model_dir).exists():
            logger.warning("DQN model not found — using untrained model")
            return
        
        checkpoints = torch.load(self.config.model_dir, map_location=self.device)
        self.online_net.load_state_dict(checkpoints["online_net"])
        self.target_net.load_state_dict(checkpoints["target_net"])
        self.epsilon    = checkpoints["epsilon"]
        self.step_count = checkpoints["step_count"]
        self._ready     = True
        logger.info("DQN model loaded")
        
    def plan(self, state: dict) -> str:
        state_tensor = self._dict_to_tensor(state)
        
        with torch.no_grad():
            q_val = self.online_net(state_tensor)
        
        if random.random() < self.epsilon:
            action_idx = random.randint(0, self.config.duel_dqn_params.action_size - 1)
            q_spread   = 0.0
        else:
            action_idx  = q_val.argmax().item()
            q_spread    = float(q_val.max() - q_val.min())
        
        action = ACTIONS[action_idx] 
        logger.info(f"DQN action: {action} (epsilon={self.epsilon:.3f})")
        return {
            "action": action, 
            "q_spread": q_spread,
            "q_values": q_val.squeeze().tolist()
        }
    
    def _dict_to_tensor(self, state: dict) -> torch.Tensor:
        n = self.config.duel_dqn_params.output_steps  # 5
        values = (
            [v for v in state.get("current_cpu",           [0.0])] +
            [v for v in state.get("current_ram",           [0.0])] +
            [v for v in state.get("current_latency",       [0.0])] +
            [v for v in state.get("current_drift",         [0.0])] +
            [v for v in state.get("predicted_cpu",         [0.0])] +
            [v for v in state.get("predicted_ram",         [0.0])] +
            [v for v in state.get("predicted_latency",     [0.0])] +
            [v for v in state.get("predicted_drift",       [0.0])]
        )
        return torch.tensor(values, dtype=torch.float32).unsqueeze(0).to(self.device)
    # ------------------------------------------------------------------------------------
    
    # ----------------------- Training DQN (simulation) -------------------------------
    def train(self, config: Simulation_Config, n_episodes: int = 1000):
        from dqn.components.simulation import SystemSimulation
        self.simulation = SystemSimulation(config)
        
        for episode in range(n_episodes):
            total_reward = self._learn_and_reward()
            
            # Decay epsilon
            self.epsilon = max(
                self.config.dqn_params.eps_min, 
                self.epsilon * self.config.dqn_params.eps_decay
            )
            
            if episode % 100 == 0:
                logger.info(f"Episode {episode}/{n_episodes} — Reward: {total_reward:.2f} — Epsilon: {self.epsilon:.3f}")

        self._save()
        self._ready = True
    
    def _learn_and_reward(self):
        state = self.simulation.reset()
        total_reward = 0
            
        for _ in range(500):
            action_idx = self._select_action(state)
            next_state, reward, done = self.simulation.step(action_idx)
                
            self.buffer.push(Experience(state, action_idx, reward, next_state, done))
            self._learn()
                
            state = next_state
            total_reward += reward
            self.step_count += 1
                
            if self.step_count % self.config.dqn_params.target_update_freq == 0:
                self.target_net.load_state_dict(self.online_net.state_dict())
                
            if done:
                break
        return total_reward
    
    
    def _select_action(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randint(0, self.config.duel_dqn_params.action_size - 1)
        
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(self.device)
        with torch.no_grad():
            return self.online_net(state_tensor).argmax().item()
    
    
    def _learn(self):
        if len(self.buffer) < self.config.dqn_params.batch_size:
            return
        
        batch       = self.buffer.sample(self.config.dqn_params.batch_size)
        states      = torch.tensor(np.array([e.state  for e in batch]), dtype=torch.float32).to(self.device)
        actions     = torch.tensor(np.array([e.action for e in batch]), dtype=torch.long).to(self.device)
        rewards     = torch.tensor(np.array([e.reward for e in batch]), dtype=torch.float32).to(self.device)
        dones       = torch.tensor(np.array([e.done   for e in batch]), dtype=torch.float32).to(self.device)
        next_states = torch.tensor(np.array([e.next_state for e in batch]), dtype=torch.float32).to(self.device)
        
        # Double DQN: online net chooses action
        with torch.no_grad():
            next_actions = self.online_net(next_states).argmax(dim=1)
            next_q       = self.target_net(next_states).gather(1, next_actions.unsqueeze(1)).squeeze()
            target_q     = rewards + self.config.dqn_params.gamma * next_q * (1 - dones)
        
        current_q = self.online_net(states).gather(1, actions.unsqueeze(1)).squeeze()
        loss      = F.smooth_l1_loss(current_q, target_q)
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.online_net.parameters(), 10)
        self.optimizer.step()
    
    def _save(self):
        torch.save({
            "online_net":  self.online_net.state_dict(),
            "target_net":  self.target_net.state_dict(),
            "epsilon":     self.epsilon,
            "step_count":  self.step_count,
        }, self.config.model_dir)
        
        logger.info(f"DQN saved -> {self.config.model_dir}")

# TRAIN

In [8]:
try:
    config = ConfigManager()
    planner = DQNPlanner(config.get_dqn_planner_config())
    planner.train(config.get_simulation_config())
except Exception as e:
    logger.error(f"Error: {e}")
    raise

[2026-05-04 06:05:51,525: INFO: __init__: yaml file: config/config.yaml loaded successfully]
[2026-05-04 06:05:51,534: INFO: __init__: yaml file: params.yaml loaded successfully]
[2026-05-04 06:05:51,536: INFO: __init__: created directory at: artifacts]
[2026-05-04 06:05:51,537: INFO: __init__: created directory at: artifacts/dqn_planner]
[2026-05-04 06:05:53,394: INFO: 2625563701: Episode 0/1000 — Reward: -151.00 — Epsilon: 0.995]
[2026-05-04 06:06:40,677: INFO: 2625563701: Episode 100/1000 — Reward: -15.00 — Epsilon: 0.603]
[2026-05-04 06:07:36,517: INFO: 2625563701: Episode 200/1000 — Reward: 105.00 — Epsilon: 0.365]
[2026-05-04 06:08:31,989: INFO: 2625563701: Episode 300/1000 — Reward: 135.00 — Epsilon: 0.221]
[2026-05-04 06:09:30,292: INFO: 2625563701: Episode 400/1000 — Reward: 170.00 — Epsilon: 0.134]
[2026-05-04 06:10:29,813: INFO: 2625563701: Episode 500/1000 — Reward: 163.00 — Epsilon: 0.081]
[2026-05-04 06:11:28,018: INFO: 2625563701: Episode 600/1000 — Reward: 189.00 — Epsi

# REFERENCE

In [16]:
import numpy as np

sim_config = config.get_simulation_config()
sim = SystemSimulation(sim_config)

planner = DQNPlanner(config.get_dqn_planner_config())
planner.load()

[2026-05-04 06:24:51,832: INFO: __init__: created directory at: artifacts/dqn_planner]
[2026-05-04 06:24:51,840: INFO: 2423171599: DQN model loaded]


In [22]:
# Test scenario 1: high drift
sim.drift = 0.8
sim.cpu = 0.2
sim.ram = 0.4
sim.latency = 0.05
state_array = sim._get_state()

state_dict = {
    "current_cpu":      [sim.cpu],
    "current_ram":      [sim.ram],
    "current_latency":  [sim.latency],
    "current_drift":    [sim.drift],
    "predicted_cpu":      list(state_array[4:9]),
    "predicted_ram":      list(state_array[9:14]),
    "predicted_latency":  list(state_array[14:19]),
    "predicted_drift":    list(state_array[19:24]),
}

result = planner.plan(state_dict)
print(f"High CPU → DQN suggests: {result['action']}")

[2026-05-04 06:25:17,689: INFO: 2423171599: DQN action: trigger_retraining (epsilon=0.010)]
High CPU → DQN suggests: trigger_retraining


In [18]:
# Test scenario 2: high CPU
sim.drift = 0.1
sim.cpu = 0.9
sim.ram = 0.7
sim.latency = 0.15
state_array = sim._get_state()

state_dict = {
    "current_cpu":      [sim.cpu],
    "current_ram":      [sim.ram],
    "current_latency":  [sim.latency],
    "current_drift":    [sim.drift],
    "predicted_cpu":      list(state_array[4:9]),
    "predicted_ram":      list(state_array[9:14]),
    "predicted_latency":  list(state_array[14:19]),
    "predicted_drift":    list(state_array[19:24]),
}

result = planner.plan(state_dict)
print(f"High CPU → DQN suggests: {result['action']}")

[2026-05-04 06:24:51,864: INFO: 2423171599: DQN action: scale_out_service (epsilon=0.010)]
High CPU → DQN suggests: scale_out_service


In [19]:
# Test scenario 3: healthy
sim.drift = 0.1
sim.cpu = 0.2
sim.ram = 0.3
sim.latency = 0.02
state_array = sim._get_state()

state_dict = {
    "current_cpu":      [sim.cpu],
    "current_ram":      [sim.ram],
    "current_latency":  [sim.latency],
    "current_drift":    [sim.drift],
    "predicted_cpu":      list(state_array[4:9]),
    "predicted_ram":      list(state_array[9:14]),
    "predicted_latency":  list(state_array[14:19]),
    "predicted_drift":    list(state_array[19:24]),
}

result = planner.plan(state_dict)
print(f"Healthy → DQN suggests: {result['action']}")

[2026-05-04 06:24:51,875: INFO: 2423171599: DQN action: do_nothing (epsilon=0.010)]
Healthy → DQN suggests: do_nothing


In [20]:
import inspect
from dqn.components.simulation import SystemSimulation
print(inspect.getfile(SystemSimulation))

scenarios = []
for _ in range(1000):
    sim.reset()
    scenarios.append('high_drift' if sim.drift > 0.3 else 'healthy')
    
from collections import Counter
print(Counter(scenarios))

/home/minh_khai/pneumonia-mlops/packages/dqn/src/dqn/components/simulation.py
Counter({'healthy': 1000})


In [21]:
# Không load lại — dùng planner hiện tại
sim.drift = 0.8
sim.cpu = 0.2
sim.ram = 0.4  
sim.latency = 0.05
state_array = sim._get_state()
state_dict = {
    "current_cpu":     [sim.cpu],
    "current_ram":     [sim.ram],
    "current_latency": [sim.latency],
    "current_drift":   [sim.drift],
    "predicted_cpu":     list(state_array[4:9]),
    "predicted_ram":     list(state_array[9:14]),
    "predicted_latency": list(state_array[14:19]),
    "predicted_drift":   list(state_array[19:24]),
}
result = planner.plan(state_dict)
print(f"High drift → {result['action']}")

[2026-05-04 06:24:51,985: INFO: 2423171599: DQN action: trigger_retraining (epsilon=0.010)]
High drift → trigger_retraining
